# Práctica 6 - Preprocesamiento de datos para Machine Learning
**Sistemas de Aprendizaje Automático · Sesión 6**   
**Dataset:** `clientes_ecommerce.csv` · **Modalidad:** individual

## Contexto
Trabajas en una empresa de comercio electrónico que quiere predecir si un cliente comprará el próximo mes. El dataset ya se exploró en las sesiones 4 y 5. Ahora debes transformarlo para que esté listo para entrenar un modelo de clasificación binaria.

**No entrenes modelos en esta práctica.** Completa los bloques `TODO` y responde las preguntas en las celdas Markdown de respuesta.

### Objetivos
- Detectar ausentes y anomalías
- Elegir estrategias de imputación 
- Codificar variables nominales y ordinales
- Comparar escalados
- Separar entrenamiento/prueba
- Construir un `Pipeline` con `ColumnTransformer` 
- Evitar data leakage
- Verificar las matrices resultantes

> Coloca `clientes_ecommerce.csv` en la misma carpeta que este notebook. Usa scikit-learn 1.2 o posterior para `sparse_output=False`.

## Parte 1 - Diagnóstico del dataset
### Ejercicio 1. Carga y estructura
Carga el CSV, muestra 10 filas, dimensiones, tipos de datos, información general y número de registros duplicados.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# TODO: cargar clientes_ecommerce.csv en df
df = pd.read_csv('clientes_ecommerce.csv')

# TODO: mostrar 10 primeras filas
print(df.head(10))

# TODO: consultar shape, dtypes, info y número de duplicados
print(df.shape)
print(df.dtypes)
df.info()
print("Filas duplicadas:", df.duplicated().sum())

   cliente_id  edad  antiguedad_meses  ingresos_anuales  gasto_mensual  \
0       20001    67             102.0           9000.00          60.73   
1       20002    28              93.0          35683.86         131.74   
2       20003    19              87.0          37152.19          55.33   
3       20004    55              79.0          40158.29          24.33   
4       20005    39               3.0          27317.06          88.78   
5       20006    45              25.0          32627.01          59.77   
6       20007    22             105.0          44101.68         109.67   
7       20008    39              90.0          44882.10          57.56   
8       20009    55              56.0          55299.19          49.94   
9       20010    38              93.0           9000.00         110.69   

   num_compras  devoluciones dispositivo    zona nivel_fidelidad  \
0            7             3       Móvil   Oeste             Oro   
1            6             1       Móvil    Este 

**Responde:** ¿Cuántas muestras y columnas hay? ¿Cuál es el target? ¿Qué columna identifica a cada cliente? ¿Hay filas duplicadas?

**Respuesta:**

- 400 muestras y 12 columnas
- `compra_proximo_mes`
- `cliente_id`
- No, hay 0 filas duplicadas

### Ejercicio 2. Valores ausentes
Calcula el número de ausentes por columna, el porcentaje y el número de filas con al menos un ausente. Completa la tabla.

In [2]:
# TODO: ausentes por columna
ausentes = df.isna().sum()
print(ausentes)

# TODO: porcentaje de ausentes por columna
porcentaje = df.isna().mean().mul(100) # O usar * 100
print(porcentaje)

# TODO: filas con algún valor ausente
filas_incompletas = df.isna().any(axis=1).sum()
print(filas_incompletas)

cliente_id            0
edad                  0
antiguedad_meses      5
ingresos_anuales      4
gasto_mensual         3
num_compras           0
devoluciones          0
dispositivo           0
zona                  0
nivel_fidelidad       0
satisfaccion          0
compra_proximo_mes    0
dtype: int64
cliente_id            0.00
edad                  0.00
antiguedad_meses      1.25
ingresos_anuales      1.00
gasto_mensual         0.75
num_compras           0.00
devoluciones          0.00
dispositivo           0.00
zona                  0.00
nivel_fidelidad       0.00
satisfaccion          0.00
compra_proximo_mes    0.00
dtype: float64
12


| Variable | Nº ausentes | Porcentaje |
|---|---:|---:|
| `antiguedad_meses` | 5 | 1.25% |
| `ingresos_anuales` | 4 | 1.00% |
| `gasto_mensual` | 3 | 0.75% |

**¿Eliminarías todas las filas incompletas? ¿Por qué?**

Respuesta: No, porque solo son 12 filas (el 3% del total) en un dataset pequeño de 400 muestras, y perderíamos información válida del resto de columnas cuando se pueden imputar fácilmente

### Ejercicio 3. Edad sospechosa
Localiza edades superiores a 120 años, muestra `cliente_id` y `edad` y consulta mínimo y máximo.

In [3]:
# TODO: mínimo y máximo de edad
print("Edad minima:", df['edad'].min())
print("Edad maxima:", df['edad'].max())

# TODO: localizar registros con edad > 120
df.loc[df["edad"] > 120, ["cliente_id", "edad"]]

Edad minima: 18
Edad maxima: 195


,cliente_id,edad
137,20138,195


**Responde:** ¿Qué harías con esa edad? ¿Por qué no deberías cambiar 195 por 95 sin evidencia?

Respuesta: La marcaría como valor ausente (`NaN`) para imputarla después con la mediana, y no la cambiaría por 95 porque no tenemos forma de saber si el fallo fue escribir 95, 19 o 59

## Parte 2 - Variables y división
### Ejercicio 4. Clasificación de columnas
Clasifica las columnas según su naturaleza.

| Tipo | Columnas |
|---|---|
| Numéricas | `edad`, `antiguedad_meses`, `ingresos_anuales`, `gasto_mensual`, `num_compras`, `devoluciones` |
| Categóricas nominales | `dispositivo`, `zona` |
| Categóricas ordinales | `nivel_fidelidad`, `satisfaccion` |
| Identificador | `cliente_id` |
| Target | `compra_proximo_mes` |

**Responde:** ¿Por qué `zona` es nominal y `nivel_fidelidad` ordinal? ¿Por qué no incluirías `cliente_id` en el modelo?

Respuesta: `zona` es nominal porque sus categorías no tienen ningún orden o jerarquía, mientras que `nivel_fidelidad` sí sigue un orden lógico de menor a mayor nivel (`Bronce < Plata < Oro < Platino`). No incluiría `cliente_id` porque es solo un identificador único sin valor predictivo que provocaría sobreajuste

### Ejercicio 5. Separar X e y
Excluye de X el identificador y el target. Codifica `No` como 0 y `Sí` como 1. Comprueba dimensiones y distribución de y.

In [4]:
# TODO: X = ... (sin cliente_id ni compra_proximo_mes)
X = df.drop(columns=['cliente_id', 'compra_proximo_mes'])

# TODO: y = ... (mapear No:0, Sí:1)
y = df['compra_proximo_mes'].map({'No': 0, 'Sí': 1})

# TODO: X.shape, y.shape, y.value_counts()
print(X.shape)
print(y.shape)
print(y.value_counts())

(400, 10)
(400,)
compra_proximo_mes
0    246
1    154
Name: count, dtype: int64


**¿Cuántas características predictoras iniciales hay?**

Respuesta: 10

### Ejercicio 6. División entrenamiento/prueba
Separa 80 % para entrenamiento y 20 % para prueba. Utiliza `random_state=42` y `stratify=y`. Comprueba tamaños.

In [5]:
# TODO: completar
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# TODO: mostrar las dimensiones de los cuatro conjuntos
print("Entretenimiento: ", X_train.shape, y_train.shape)
print("Test: ", X_test.shape, y_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

Entretenimiento:  (320, 10) (320,)
Test:  (80, 10) (80,)
compra_proximo_mes
0    0.615625
1    0.384375
Name: proportion, dtype: float64
compra_proximo_mes
0    0.6125
1    0.3875
Name: proportion, dtype: float64


**Responde:** ¿Cuántas muestras hay en cada conjunto? ¿Para qué sirve `stratify=y`? ¿Por qué dividimos antes de ajustar transformaciones?

Respuesta: Hay 320 muestras en entrenamiento (80%) y 80 en prueba (20%). `stratify=y` mantiene la misma proporción de clases (aprox. 61.5% de 0 y 38.5% de 1) en ambos conjuntos. Dividimos antes para evitar fuga de datos (*data leakage*), asegurando que el conjunto de prueba no influya en el cálculo de medianas, modas o escalados

## Parte 3 - Imputación y encoding
### Ejercicio 7. Validación de edad
Sobre **copias** de `X_train` y `X_test`, sustituye edades superiores a 120 por `np.nan`. No imputes todavía: eso se hará en el pipeline. El umbral es una regla didáctica de validación del dominio.

In [6]:
X_train = X_train.copy()
X_test = X_test.copy()

# TODO: marcar edades > 120 como NaN en ambos conjuntos
print(X_train.loc[X_train['edad'] > 120, ["edad"]])
X_train.loc[X_train['edad'] > 120, ["edad"]] = np.nan

print(X_test.loc[X_test['edad'] > 120, ["edad"]])
X_test.loc[X_test['edad'] > 120, ["edad"]] = np.nan

# TODO: contar ausentes en edad en entrenamiento y prueba
print("Ausentes edad train:", X_train['edad'].isna().sum())
print("Ausentes edad test:", X_test['edad'].isna().sum())

     edad
137   195
Empty DataFrame
Columns: [edad]
Index: []
Ausentes edad train: 1
Ausentes edad test: 0


### Ejercicio 8. Estrategias de imputación
Utilizaremos mediana para numéricas y moda para categóricas. Crea ambos objetos `SimpleImputer` **sin ajustarlos todavía**.

In [7]:
# TODO: imputer_numerico = SimpleImputer(...)
"""
strategy:
- Media: mean
- Mediana: median
- Moda: most_frequent
- Valor constante: constant => fill_value=x
"""
imputer_numerico = SimpleImputer(strategy="median")

# TODO: imputer_categorico = SimpleImputer(...)
imputer_categorico = SimpleImputer(strategy="most_frequent")

| Variable | Estrategia | Justificación |
|---|---|---|
| edad | Mediana | Es robusta frente a posibles valores extremos o asimetrías |
| antiguedad_meses | Mediana | Evita que valores altos o dispersos distorsionen el valor imputado |
| ingresos_anuales | Mediana | Mantiene un valor central representativo sin verse afectada por colas |
| gasto_mensual | Mediana | La distribución tiene asimetría positiva y outliers altos, donde la media se desplaza |
| dispositivo | Moda | Al ser categórica nominal no tiene media ni mediana, se usa el valor más frecuente |
| zona | Moda | Es categórica nominal y se rellena con la categoría más habitual |

**¿Por qué no ajustar el imputador con entrenamiento y prueba a la vez?**

Respuesta: Porque produciría fuga de datos (*data leakage*), filtrando información estadística del conjunto de prueba dentro del entrenamiento y falseando la evaluación real del modelo

### Ejercicio 9. One-Hot Encoding
Prueba `OneHotEncoder` con un DataFrame independiente. Configura `handle_unknown="ignore"` y `sparse_output=False`. Muestra matriz y nombres de columnas.

In [8]:
ejemplo = pd.DataFrame({
    "dispositivo": ["Móvil", "Ordenador", "Tablet", "Móvil", "Ordenador", "Tablet", "Móvil"]
})

# TODO: encoder_nominal = OneHotEncoder(...)
encoder_nominal = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# TODO: aplicar fit_transform a ejemplo
matriz_ejemplo = encoder_nominal.fit_transform(ejemplo)
print(matriz_ejemplo)

# TODO: consultar get_feature_names_out()
print(encoder_nominal.get_feature_names_out())
pd.DataFrame(matriz_ejemplo, columns=encoder_nominal.get_feature_names_out())


[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]
['dispositivo_Móvil' 'dispositivo_Ordenador' 'dispositivo_Tablet']


,dispositivo_Móvil,dispositivo_Ordenador,dispositivo_Tablet
0,1.0,0.0,0.0
1,0.0,1.0,0.0
2,0.0,0.0,1.0
3,1.0,0.0,0.0
4,0.0,1.0,0.0
5,0.0,0.0,1.0
6,1.0,0.0,0.0


**Responde:** ¿Cuántas columnas se generan? ¿Qué representan? ¿Por qué no codificar Móvil=0, Ordenador=1 y Tablet=2 como categorías ordenadas?

Respuesta: Se generan 3 columnas binarias (`dispositivo_Móvil`, `dispositivo_Ordenador`, `dispositivo_Tablet`) que indican con 1 o 0 la presencia de cada categoría. No se codifican como 0, 1 y 2 porque son categorías nominales sin orden y el modelo asumiría falsamente que Tablet es mayor que Móvil o que hay una distancia numérica entre ellos

### Ejercicio 10. Codificación ordinal
Configura `OrdinalEncoder` para el orden Bronce, Plata, Oro, Platino. Comprueba su funcionamiento con un ejemplo independiente.

In [9]:
ejemplo_fidelidad = pd.DataFrame({
    "nivel_fidelidad": ["Oro", "Bronce", "Platino", "Plata"]
})

print(ejemplo_fidelidad['nivel_fidelidad'])

# TODO: encoder_ordinal = OrdinalEncoder(categories=[[ ... ]])
encoder_ordinal = OrdinalEncoder(categories=[["Bronce", "Plata", "Oro", "Platino"]])

# TODO: transformar ejemplo_fidelidad y mostrar el resultado
ejemplo_fidelidad['nivel_fidelidad'] = encoder_ordinal.fit_transform(ejemplo_fidelidad[['nivel_fidelidad']])
print(ejemplo_fidelidad)

0        Oro
1     Bronce
2    Platino
3      Plata
Name: nivel_fidelidad, dtype: str
   nivel_fidelidad
0              2.0
1              0.0
2              3.0
3              1.0


**Responde:** ¿Qué representan los códigos? ¿La distancia entre niveles consecutivos es necesariamente igual?

Respuesta: ...

## Parte 4 - Escalado
### Ejercicio 11. Min-Max frente a Z-score
Aplica ambos escaladores al siguiente ejemplo. Redondea a tres decimales y completa la tabla.

In [10]:
datos = pd.DataFrame({"valor": [10, 20, 30, 40, 50]})
print(datos)

# TODO: crear MinMaxScaler y StandardScaler
minmax = MinMaxScaler()
zscore = StandardScaler()

# TODO: transformar datos con cada uno
print(minmax.fit_transform(datos))
print(zscore.fit_transform(datos))

comparacion = {
    "datos":datos['valor'],
    "minmax":minmax.fit_transform(datos).ravel(),
    "zscore":zscore.fit_transform(datos).ravel()
}

print(pd.DataFrame(comparacion))

# TODO: mostrar tabla comparativa

   valor
0     10
1     20
2     30
3     40
4     50
[[0.  ]
 [0.25]
 [0.5 ]
 [0.75]
 [1.  ]]
[[-1.41421356]
 [-0.70710678]
 [ 0.        ]
 [ 0.70710678]
 [ 1.41421356]]
   datos  minmax    zscore
0     10    0.00 -1.414214
1     20    0.25 -0.707107
2     30    0.50  0.000000
3     40    0.75  0.707107
4     50    1.00  1.414214


| Original | Min-Max | StandardScaler |
|---:|---:|---:|
| 10 | 0.000 | -1.414 |
| 20 | 0.250 | -0.707 |
| 30 | 0.500 | 0.000 |
| 40 | 0.750 | 0.707 |
| 50 | 1.000 | 1.414 |

**Responde:** ¿Qué rango da Min-Max en este ejemplo? ¿Qué valor toma la media tras estandarizar? ¿Pueden aparecer valores negativos? ¿Qué escalado elegirías para un algoritmo basado en distancias y por qué?

Respuesta: Min-Max da un rango entre 0 y 1. Tras estandarizar con `StandardScaler` la media pasa a ser 0 y sí pueden aparecer valores negativos (para datos por debajo de la media). Para algoritmos basados en distancias (como KNN) elegiría `StandardScaler` porque centra los datos y es menos sensible a que un outlier aplaste el resto de valores

### Ejercicio 12. Variables que escalaremos
Para esta práctica trataremos `satisfaccion` como numérica, aunque conceptualmente es una escala ordinal de 1 a 5. Es una simplificación que deberá revisarse en otros modelos.

In [11]:
numericas = [
    "edad", "antiguedad_meses", "ingresos_anuales",
    "gasto_mensual", "num_compras", "devoluciones", "satisfaccion"
]
nominales = ["dispositivo", "zona"]
ordinales = ["nivel_fidelidad"]

# TODO: comprobar que las columnas existen en X_train
columnas_totales = numericas + nominales + ordinales
print(all(col in X_train.columns for col in columnas_totales))

True


**Responde:** ¿Qué variables tienen escalas más distintas? ¿Es obligatorio escalar para cualquier algoritmo? ¿Qué particularidad tiene `satisfaccion`?

Respuesta: `ingresos_anuales` (que llega hasta más de 80.000) frente a `devoluciones` (de 0 a 5) o `satisfaccion` (de 1 a 5). No es obligatorio para todos los algoritmos, ya que los modelos basados en árboles (como árboles de decisión o Random Forest) no dependen de la escala. `satisfaccion` es conceptualmente una variable categórica ordinal (de 1 a 5), aunque aquí se trata como numérica por simplificación

## Parte 5 - Pipeline completo
### Ejercicio 13. Pipeline numérico
Construye un `Pipeline` que impute con mediana y después estandarice.

In [12]:
pipeline_numerico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

**¿Por qué imputamos antes de escalar en este pipeline?**

Respuesta: Porque `StandardScaler` necesita calcular la media y la desviación estándar sobre valores numéricos completos y no puede operar si hay valores nulos (`NaN`)

### Ejercicio 14. Pipeline nominal
Imputa con moda y después aplica One-Hot Encoding (`handle_unknown="ignore"`, `sparse_output=False`).

In [13]:
pipeline_nominal = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

### Ejercicio 15. Pipeline ordinal
Imputa con moda y codifica Bronce → Plata → Oro → Platino. Gestiona categorías desconocidas con `handle_unknown="use_encoded_value"` y `unknown_value=-1`.

In [14]:
pipeline_ordinal = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        categories=[["Bronce", "Plata", "Oro", "Platino"]],
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

### Ejercicio 16. ColumnTransformer
Combina los tres pipelines para aplicar cada transformación únicamente a sus columnas.

In [15]:
preprocesador = ColumnTransformer(transformers=[
    ("num", pipeline_numerico, numericas),
    ("nom", pipeline_nominal, nominales),
    ("ord", pipeline_ordinal, ordinales)
])

### Ejercicio 17. Transformar entrenamiento y prueba
Ajusta el preprocesador **solo** con entrenamiento y aplícalo a prueba sin reajustarlo.

In [16]:
# TODO: X_train_preparado = preprocesador.fit_transform(...)
X_train_preparado = preprocesador.fit_transform(X_train)

# TODO: X_test_preparado = preprocesador.transform(...)
X_test_preparado = preprocesador.transform(X_test)

**Responde:** ¿Por qué no usar `fit_transform()` en prueba? ¿Qué aprende `fit()`? ¿Qué ocurre si aparece una categoría nominal nueva en prueba?

Respuesta: No se usa `fit_transform()` en prueba para evitar fuga de datos y aplicar exactamente los mismos parámetros aprendidos en entrenamiento. `fit()` aprende las medianas, modas, categorías únicas, medias y desviaciones estándar de `X_train`. Si aparece una categoría nominal nueva en prueba, `OneHotEncoder` con `handle_unknown="ignore"` la ignora sin dar error y pone ceros en todas sus columnas